# SANPO Bucket Streaming Kinetic Replay (Colab)

This notebook:
- mounts Google Drive once
- reads your processed JSONL session from Drive
- streams SANPO RGB frames directly from `gs://gresearch/sanpo_dataset/v0/sanpo-synthetic/`
- recomputes per-object kinetic scores using your custom formula
- writes an annotated MP4 back to Drive

No zip re-upload is required after the first Drive setup. Put the helper pack and your JSONL file in Drive, then open this notebook from Drive or Colab.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Video, display
from google.colab import drive


drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
PACK_DIR = DRIVE_ROOT / 'Capstone' / 'RandomColabStuff'
ZIP_PATH = PACK_DIR / 'sanpo_kinetic_replay_colab.zip'

if ZIP_PATH.exists() and not (PACK_DIR / 'sanpo_bucket_replay.py').exists():
    PACK_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Unzipping pack from {ZIP_PATH} to {PACK_DIR}')
    !unzip -o -q "{ZIP_PATH}" -d "{PACK_DIR}"

if str(PACK_DIR) not in sys.path:
    sys.path.insert(0, str(PACK_DIR))

from sanpo_bucket_replay import (
    DEFAULT_FORMULA,
    fetch_frame_image,
    get_anonymous_client,
    list_sessions,
    parse_jsonl_session,
    render_replay_from_bucket,
)

print('Drive root:', DRIVE_ROOT)
print('Pack dir  :', PACK_DIR)

In [ ]:
# ---------- User Config ----------
PROJECT_ROOT = PACK_DIR

# New processed JSONL schema file (one record per frame with an `objects` array).
# /content/drive/MyDrive/Capstone/RandomColabStuff/sanpo_input/<session_file>.jsonl
JSONL_PATH = DRIVE_ROOT / 'Capstone' / 'RandomColabStuff' / 'sanpo_input' / '-5OCPnbrwJdu3jH70ieU7pUiFsOJQoeG.jsonl'

# Default session id comes from the JSONL filename. Override manually if needed.
SESSION_ID = JSONL_PATH.stem

# Data pipeline uses frame step = 3 by design.
EXPECTED_FRAME_STRIDE = 3

RGB_FRAMES_DIR = PROJECT_ROOT / f'data/sanpo/raw/{SESSION_ID}/camera_head/left/video_frames'
DESCRIPTION_JSON = PROJECT_ROOT / f'data/sanpo/raw/{SESSION_ID}/description.json'
OUTPUT_MP4 = DRIVE_ROOT / 'Capstone' / 'RandomColabStuff' / 'sanpo_output' / f'{JSONL_PATH.stem}_{SESSION_ID[:8]}_kinetic_replay_bbox.mp4'

# Render options
SCALE = 0.60
DRAW_TOP_N = 8
OUTPUT_FPS = None  # None => infer from session fps / frame stride

# ---------- Custom Kinetic Formula Parameters ----------
# K = severity(class) * direction_weight(position) * |velocity|^VELOCITY_EXP / max(distance, EPS_DISTANCE)^DISTANCE_EXP
EPS_DISTANCE = 0.5
VELOCITY_EXP = 2.0
DISTANCE_EXP = 1.0
UNKNOWN_DISTANCE_SCORE = 0.0
DEFAULT_SEVERITY = 1.0

AHEAD_WEIGHT = 1.00
LATERAL_WEIGHT = 0.85
FAR_LATERAL_WEIGHT = 0.70
REAR_WEIGHT = 0.60
UNKNOWN_DIRECTION_WEIGHT = 0.75

SEVERITY_BY_CLASS = {
    'person': 1.2,
    'bicycle': 1.4,
    'motorcycle': 1.8,
    'car': 2.0,
    'bus': 2.5,
    'truck': 2.5,
    'unlabeled_obstacle': 1.3,
    'fire hydrant': 0.8,
    'shadow': 0.2,
    'boat': 0.5,
}

print(f'JSONL:       {JSONL_PATH}')
print(f'SESSION_ID:  {SESSION_ID}')
print(f'OUTPUT MP4:  {OUTPUT_MP4}')

In [ ]:
def direction_weight_from_label(position_label: str) -> float:
    p = str(position_label or 'unknown').lower()
    if 'ahead' in p:
        return float(AHEAD_WEIGHT)
    if 'far-left' in p or 'far-right' in p:
        return float(FAR_LATERAL_WEIGHT)
    if 'left' in p or 'right' in p:
        return float(LATERAL_WEIGHT)
    if 'behind' in p:
        return float(REAR_WEIGHT)
    return float(UNKNOWN_DIRECTION_WEIGHT)


def kinetic_score_from_object(obj: dict) -> float:
    distance_m = obj.get('distance_m', None)
    velocity_ms = float(obj.get('velocity_ms', 0.0) or 0.0)
    cls = str(obj.get('class', 'unknown'))

    if distance_m is None:
        return float(UNKNOWN_DISTANCE_SCORE)

    severity = float(SEVERITY_BY_CLASS.get(cls, DEFAULT_SEVERITY))
    d = max(float(distance_m), float(EPS_DISTANCE))
    v = abs(float(velocity_ms))

    return float(
        severity * direction_weight_from_label(obj.get('position_label', 'unknown'))
        * (v ** float(VELOCITY_EXP)) / (d ** float(DISTANCE_EXP))
    )


def load_jsonl_rows(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            if isinstance(row, dict) and 'frame_id' in row and 'objects' in row:
                rows.append(row)
    if not rows:
        raise RuntimeError('No valid rows in JSONL.')
    return rows


def infer_stride_from_rows(rows: list[dict]) -> int:
    frame_ids = [int(r['frame_id']) for r in rows]
    if len(frame_ids) < 2:
        return 1
    diffs = [b - a for a, b in zip(frame_ids[:-1], frame_ids[1:]) if b > a]
    if not diffs:
        return 1
    return max(1, int(round(float(np.median(np.array(diffs))))))

In [ ]:
# -------- Runtime / Render Config --------
BUCKET_NAME = 'gresearch'

# Auto-detect location from these candidates (works for both real + synthetic).
BASE_PREFIX_CANDIDATES = [
    'sanpo_dataset/v0/sanpo-real',
    'sanpo_dataset/v0/sanpo-synthetic',
]
CAMERA_CANDIDATES = ['camera_head', 'camera_chest']
SIDE_CANDIDATES = ['left', 'right']

# Resolved stream values (set automatically in next cell).
BASE_PREFIX = BASE_PREFIX_CANDIDATES[0]
CAMERA = CAMERA_CANDIDATES[0]
SIDE = SIDE_CANDIDATES[0]

CACHE_DIR = DRIVE_ROOT / 'Capstone' / 'RandomColabStuff' / 'sanpo_frame_cache'

# Build formula dict from config values in Cell 3
FORMULA = dict(DEFAULT_FORMULA)
FORMULA['velocity_exp'] = float(VELOCITY_EXP)
FORMULA['distance_exp'] = float(DISTANCE_EXP)
FORMULA['eps_distance'] = float(EPS_DISTANCE)
FORMULA['unknown_distance_score'] = float(UNKNOWN_DISTANCE_SCORE)
FORMULA['default_severity'] = float(DEFAULT_SEVERITY)

FORMULA['ahead_weight'] = float(AHEAD_WEIGHT)
FORMULA['lateral_weight'] = float(LATERAL_WEIGHT)
FORMULA['far_lateral_weight'] = float(FAR_LATERAL_WEIGHT)
FORMULA['rear_weight'] = float(REAR_WEIGHT)
FORMULA['unknown_direction_weight'] = float(UNKNOWN_DIRECTION_WEIGHT)

FORMULA['severity_by_class'] = {
    **DEFAULT_FORMULA['severity_by_class'],
    **SEVERITY_BY_CLASS,
}

print('SESSION_ID =', SESSION_ID)
print('CACHE_DIR  =', CACHE_DIR)
print('OUTPUT_MP4 =', OUTPUT_MP4)
print('Base prefix candidates:', BASE_PREFIX_CANDIDATES)
print('Camera candidates     :', CAMERA_CANDIDATES)
print('Side candidates       :', SIDE_CANDIDATES)

In [ ]:
jsonl_rows = load_jsonl_rows(JSONL_PATH)
frame_objects = parse_jsonl_session(JSONL_PATH)
frame_ids = sorted(frame_objects.keys())

all_objects = [o for objs in frame_objects.values() for o in objs]
unknown_dist = sum(1 for o in all_objects if o.distance_m is None)
observed_stride = infer_stride_from_rows(jsonl_rows)

print('Frames in JSONL:', len(frame_ids))
print('Frame id range :', frame_ids[0], '->', frame_ids[-1])
print('Object entries :', len(all_objects))
print('Unknown dist % :', round(100.0 * unknown_dist / max(1, len(all_objects)), 2))
print('Observed stride:', observed_stride)
if observed_stride != EXPECTED_FRAME_STRIDE:
    print(f'WARNING: expected stride {EXPECTED_FRAME_STRIDE}, got {observed_stride}')

In [ ]:
client = get_anonymous_client()
print('Anonymous SANPO client created.')

# -------- Resolve session stream location (prefix/camera/side) --------
probe_frame_ids = sorted(frame_objects.keys())[:20]
resolved = None
for candidate_prefix in BASE_PREFIX_CANDIDATES:
    for candidate_camera in CAMERA_CANDIDATES:
        for candidate_side in SIDE_CANDIDATES:
            hit = None
            for fid in probe_frame_ids:
                hit = fetch_frame_image(
                    client=client,
                    session_id=SESSION_ID,
                    frame_id=fid,
                    cache_dir=CACHE_DIR,
                    bucket_name=BUCKET_NAME,
                    base_prefix=candidate_prefix,
                    camera=candidate_camera,
                    side=candidate_side,
                )
                if hit is not None:
                    break
            if hit is not None:
                resolved = (candidate_prefix, candidate_camera, candidate_side, fid)
                break
        if resolved is not None:
            break
    if resolved is not None:
        break

if resolved is None:
    tried = [
        f"{bp}/{SESSION_ID}/{cam}/{side}/video_frames/<frame>.png"
        for bp in BASE_PREFIX_CANDIDATES
        for cam in CAMERA_CANDIDATES
        for side in SIDE_CANDIDATES
    ]
    raise RuntimeError(
        'Could not resolve stream location for this session. Tried:\n - ' + '\n - '.join(tried)
    )

BASE_PREFIX, CAMERA, SIDE, matched_fid = resolved
print('Resolved stream path:')
print('  BASE_PREFIX =', BASE_PREFIX)
print('  CAMERA      =', CAMERA)
print('  SIDE        =', SIDE)
print('  Matched frame id:', matched_fid)

# Optional: show a few session ids from the resolved dataset prefix
print('Sample SANPO sessions from resolved prefix:')
for sid in list_sessions(client, bucket_name=BUCKET_NAME, base_prefix=BASE_PREFIX, limit=5):
    print(' -', sid)

# -------- Debug View: 10th JSONL record --------
DEBUG_RECORD_INDEX = 10  # 1-based index in JSONL order

if len(jsonl_rows) < DEBUG_RECORD_INDEX:
    raise RuntimeError(f'JSONL has only {len(jsonl_rows)} rows; cannot inspect record {DEBUG_RECORD_INDEX}.')

debug_row = jsonl_rows[DEBUG_RECORD_INDEX - 1]
debug_frame_id = int(debug_row['frame_id'])
debug_objects = list(debug_row.get('objects', []))

img = fetch_frame_image(
    client=client,
    session_id=SESSION_ID,
    frame_id=debug_frame_id,
    cache_dir=CACHE_DIR,
    bucket_name=BUCKET_NAME,
    base_prefix=BASE_PREFIX,
    camera=CAMERA,
    side=SIDE,
)
if img is None:
    raise RuntimeError(
        f'Could not fetch frame {debug_frame_id} from session {SESSION_ID} '
        f'using {BASE_PREFIX}/{CAMERA}/{SIDE}.'
    )

disp = img.copy()
rows = []

for obj in debug_objects:
    score = kinetic_score_from_object(obj)
    obj_id = str(obj.get('object_id', 'unknown'))
    cls = str(obj.get('class', 'unknown'))
    conf = obj.get('confidence', None)
    dist = obj.get('distance_m', None)
    vel = float(obj.get('velocity_ms', 0.0) or 0.0)
    bearing = obj.get('bearing_deg', None)
    pos = str(obj.get('position_label', 'unknown'))
    bbox = obj.get('bbox_xyxy', None)

    rows.append({
        'object_id': obj_id,
        'track_id': obj.get('track_id', None),
        'class': cls,
        'distance_m': dist,
        'velocity_ms': vel,
        'bearing_deg': bearing,
        'position_label': pos,
        'confidence': conf,
        'kinetic_score': round(score, 6),
        'bbox_xyxy': bbox,
    })

    color = (0, 255, 255) if score > 0 else (160, 160, 160)
    if isinstance(bbox, list) and len(bbox) == 4:
        x1, y1, x2, y2 = [int(round(float(v))) for v in bbox]
        h, w = disp.shape[:2]
        x1 = max(0, min(w - 1, x1))
        y1 = max(0, min(h - 1, y1))
        x2 = max(0, min(w - 1, x2))
        y2 = max(0, min(h - 1, y2))
        if x2 <= x1:
            x2 = min(w - 1, x1 + 1)
        if y2 <= y1:
            y2 = min(h - 1, y1 + 1)
        cv2.rectangle(disp, (x1, y1), (x2, y2), color, 2)
        label = f"{obj_id} {cls} K*={score:.3f}"
        if conf is not None:
            label += f" c={float(conf):.2f}"
        ly = y1 - 8 if y1 > 24 else min(h - 8, y1 + 18)
        cv2.putText(disp, label, (x1, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

print(f"Debug record #{DEBUG_RECORD_INDEX}: frame_id={debug_frame_id}, objects={len(debug_objects)}")
display(pd.DataFrame(rows))

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(disp, cv2.COLOR_BGR2RGB))
plt.title(f'Session {SESSION_ID} | Frame {debug_frame_id} | Record #{DEBUG_RECORD_INDEX}')
plt.axis('off')
plt.show()

In [ ]:
summary = render_replay_from_bucket(
    client=client,
    session_id=SESSION_ID,
    frame_objects=frame_objects,
    output_mp4=OUTPUT_MP4,
    formula_cfg=FORMULA,
    cache_dir=CACHE_DIR,
    draw_top_n=DRAW_TOP_N,
    scale=SCALE,
    output_fps=OUTPUT_FPS,
    bucket_name=BUCKET_NAME,
    base_prefix=BASE_PREFIX,
    camera=CAMERA,
    side=SIDE,
)

print(summary)

In [ ]:
display(Video(str(OUTPUT_MP4), embed=True, html_attributes='controls loop'))